In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import (
    LoraConfig,
    PeftModel,
    TaskType,
    get_peft_model,
)
from trl import DataCollatorForCompletionOnlyLM, SFTConfig, SFTTrainer
from torch.utils.data import DataLoader
import torch

import numpy as np
import random

SEED = 42

def seed_everything(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


seed_everything(42)
PAD_TOKEN = "<|pad|>"
seq_length = 1800

OUTPUT_DIR ="model_training"

In [2]:
from huggingface_hub import notebook_login
notebook_login()

In [3]:
# Define your saved path
model_path = "meta-llama/Llama-3.2-1B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_path)

tokenizer.add_special_tokens({"pad_token": PAD_TOKEN})
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(model_path,)


model.resize_token_embeddings(len(tokenizer), pad_to_multiple_of=8)

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


Embedding(128264, 2048)

In [4]:
# Configure LoRA
lora_config = LoraConfig(
    r=64,                            # rank
    lora_alpha=128,
    target_modules=[
        "self_attn.q_proj",
        "self_attn.k_proj",
        "self_attn.v_proj",
        "self_attn.o_proj",
        "mlp.gate_proj",
        "mlp.up_proj",
        "mlp.down_proj",],      # T5 uses "q", "v" in attention
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

In [5]:
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 45,088,768 || all params: 1,280,919,552 || trainable%: 3.5200


In [6]:
model.print_trainable_parameters()

trainable params: 45,088,768 || all params: 1,280,919,552 || trainable%: 3.5200


In [7]:
from datasets import load_dataset

dataset = load_dataset("json", data_files={
    "train": "train.json",
    "validation": "dev.json",
    "test": "test.json"
})
dataset

DatasetDict({
    train: Dataset({
        features: ['response', 'query', 'text'],
        num_rows: 18728
    })
    validation: Dataset({
        features: ['response', 'query', 'text'],
        num_rows: 1021
    })
    test: Dataset({
        features: ['response', 'query', 'text'],
        num_rows: 1045
    })
})

In [8]:
response_template = "<|end_header_id|>"
collator = DataCollatorForCompletionOnlyLM(response_template, tokenizer=tokenizer)

examples = [dataset["train"][0]["text"]]
encodings = [tokenizer(e) for e in examples]

dataloader = DataLoader(encodings, collate_fn=collator, batch_size=8)

In [9]:
sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    dataset_text_field="text",
    max_seq_length=seq_length,
    num_train_epochs=5,
    per_device_train_batch_size=6,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=6,
    optim="adamw_torch_fused",
    eval_strategy="steps",
    eval_steps=100,
    save_steps=200,
    logging_steps=10,
    learning_rate=2e-4,
    bf16=True,
    save_strategy="steps",
    warmup_ratio=0.1,
    save_total_limit=2,
    lr_scheduler_type="constant",
    report_to="none",
    dataset_kwargs={
        "add_special_tokens": False,  # We template with special tokens
        "append_concat_token": False,  # No need to add additional separator token
    },
    seed=SEED,
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    data_collator=collator,
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [ ]:
trainer.train()

FileNotFoundError: [Errno 2] No such file or directory: './model_training/trainer_state.json'

In [17]:
NEW_MODEL = "./llama-3.2-1B-mcq-gen-finetuned"

In [ ]:
# Save the LoRA adapter first
trainer.save_model(NEW_MODEL)

In [18]:
# Load base model (FRESH, not quantized for merging)
base_model = AutoModelForCausalLM.from_pretrained(
    model_path,
    torch_dtype=torch.float16,
    device_map="auto",
)

# Prepare tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_path)  # ✅ Load from base, not NEW_MODEL
tokenizer.add_special_tokens({"pad_token": PAD_TOKEN})
tokenizer.padding_side = "right"

# Resize embeddings to match training
base_model.resize_token_embeddings(len(tokenizer), pad_to_multiple_of=8)

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Embedding(128264, 2048)

In [19]:
# Load LoRA adapter and merge
model_with_adapter = PeftModel.from_pretrained(base_model, NEW_MODEL)
merged_model = model_with_adapter.merge_and_unload()
print("Model merged and saved.")

Model merged and saved.


In [20]:
# Save merged model locally
MERGED_MODEL_DIR = "./llama-3.2-1B-mcq-gen-finetuned-merged"
merged_model.save_pretrained(MERGED_MODEL_DIR)
tokenizer.save_pretrained(MERGED_MODEL_DIR)
print("Model merged and saved.")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model merged and saved.


# Create repo first


In [22]:
import os
merged_model.push_to_hub(
    "sinister007/llama-3.2-1B-mcq-gen-finetuned",
    token=os.getenv("HF_TOKEN")
)
tokenizer.push_to_hub(
    "sinister007/llama-3.2-1B-mcq-gen-finetuned",
    token=os.getenv("HF_TOKEN")
)

print("Model and tokenizer uploaded to sinister007/llama-3.2-1B-mcq-gen-finetuned")

README.md:   0%|          | 0.00/31.0 [00:00<?, ?B/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Model and tokenizer uploaded to sinister007/llama-3.2-1B-mcq-gen-finetuned


In [ ]:
login(token=HF_TOKEN)

api = HfApi()

# Upload checkpoints to a 'checkpoints' folder in your main repo
api.upload_folder(
    folder_path="./model_training",
    repo_id="sinister007/llama-3.2-1B-mcq-gen-finetuned",
    path_in_repo="checkpoints",  # Creates a 'checkpoints' subfolder
    repo_type="model",
    commit_message="Upload training checkpoints",
)

print("✅ Checkpoints uploaded to sinister007/llama-3.2-1B-mcq-gen-finetuned/checkpoints")